In [1]:
!pip install pydriller pandas

You should consider upgrading via the '/Users/srajansharma/Developer/Projects/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
from pydriller import Repository
import re

In [4]:

REPO_PATH = "camel" #cloned repo in my local machine

ISSUE_IDS = [
    "CAMEL-180",
    "CAMEL-321",
    "CAMEL-1818",
    "CAMEL-3214",
    "CAMEL-18065"
]

def is_issue_commit(message, issue_ids):
    """Check if commit message contains any of the issue IDs"""
    return any(re.search(issue_id, message) for issue_id in issue_ids)


def main():
    unique_commit_hashes = set()
    selected_commits = []

    print("Scanning repository... (this may take a few minutes)")

    # Traverse commits
    for commit in Repository(REPO_PATH).traverse_commits():

        # Skip merge commits
        if len(commit.parents) > 1:
            continue

        # Check if commit matches any issue ID
        if is_issue_commit(commit.msg, ISSUE_IDS):

            if commit.hash not in unique_commit_hashes:
                unique_commit_hashes.add(commit.hash)
                selected_commits.append(commit)

    total_commits = len(selected_commits)

    if total_commits == 0:
        print("No matching commits found.")
        return


    unique_files = set()
    dmm_scores = []

    for commit in selected_commits:

        # Collect changed files
        for file in commit.modified_files:
            if file.change_type.name in ["ADD", "MODIFY", "DELETE"]:

                if file.new_path:
                    unique_files.add(file.new_path)
                elif file.old_path:
                    unique_files.add(file.old_path)

        # Compute DMM score
        if (
            commit.dmm_unit_size is not None and
            commit.dmm_unit_complexity is not None and
            commit.dmm_unit_interfacing is not None
        ):
            dmm = (
                commit.dmm_unit_size +
                commit.dmm_unit_complexity +
                commit.dmm_unit_interfacing
            ) / 3

            dmm_scores.append(dmm)


    avg_files_changed = len(unique_files) / total_commits
    avg_dmm = sum(dmm_scores) / len(dmm_scores) if dmm_scores else 0

    print("\n=== RESULTS ===")
    print(f"Total commits analyzed: {total_commits}")
    print(f"Average number of files changed: {avg_files_changed:.2f}")
    print(f"Average DMM metrics: {avg_dmm:.2f}")



if __name__ == "__main__":
    main()

Scanning repository... (this may take a few minutes)

=== RESULTS ===
Total commits analyzed: 169
Average number of files changed: 3.75
Average DMM metrics: 0.60
